# 🚀 Zyro Dynamics HR Help Desk — RAG Challenge
**Optimized Submission | BGE-base Embeddings | Multi-Query MMR | LangSmith Tracing**

Scoring: Q01–Q10 (in-scope, 8 pts each) + Q11–Q15 (out-of-scope refusal, 4 pts each) = 100 pts

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q langchain langchain-community langchain-text-splitters \
    langchain-huggingface langchain-groq langchain-core langsmith \
    faiss-cpu pypdf sentence-transformers huggingface_hub
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Imports & API keys ────────────────────────────────────────────────
import os, re, base64
import pandas as pd
from kaggle_secrets import UserSecretsClient

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

secrets = UserSecretsClient()
os.environ['GROQ_API_KEY']         = secrets.get_secret('GROQ_API_KEY')
os.environ['LANGCHAIN_API_KEY']    = secrets.get_secret('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT']    = 'zyro-rag-challenge'

print('✅ API keys loaded')
print(f'   GROQ key: ...{os.environ["GROQ_API_KEY"][-6:]}')
print(f'   LangSmith tracing: ON → project: zyro-rag-challenge')

In [ ]:
# ── Cell 3: TODO — Load HR PDFs ───────────────────────────────────────────────
import glob

# Auto-detect the PDF directory from Kaggle dataset
search_paths = [
    '/kaggle/input/zyro-dynamics-hr-policies',
    '/kaggle/input/zyro-hr-policies',
    '/kaggle/input',
]
pdf_files = []
DOCS_DIR  = None
for p in search_paths:
    found = glob.glob(f'{p}/**/*.pdf', recursive=True)
    if found:
        pdf_files = found
        DOCS_DIR  = p if os.path.isdir(p) else os.path.dirname(found[0])
        break

if not pdf_files:
    raise FileNotFoundError('No PDFs found! Check your Kaggle dataset is attached.')

print(f'📄 Found {len(pdf_files)} PDF files in: {DOCS_DIR}')
for f in sorted(pdf_files):
    print(f'   {os.path.basename(f)}')

loader    = PyPDFDirectoryLoader(DOCS_DIR)
documents = loader.load()
print(f'\n✅ Loaded {len(documents)} pages from {len(pdf_files)} documents')

In [ ]:
# ── Cell 4: TODO — Chunk documents ───────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=200,
    separators=['\n\n', '\n', '. ', '! ', '? ', '; ', ', ', ' ', ''],
    add_start_index=True,
)
chunks = splitter.split_documents(documents)
print(f'✅ Created {len(chunks)} chunks  (size=700, overlap=200)')

In [ ]:
# ── Cell 5: TODO — Build FAISS vector store ───────────────────────────────────
print('⏳ Loading BAAI/bge-base-en-v1.5 embedding model…')
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-base-en-v1.5',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

vectorstore    = FAISS.from_documents(chunks, embeddings)
base_retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 6, 'fetch_k': 20, 'lambda_mult': 0.6},
)
print('✅ FAISS vector store built with MMR retrieval')

In [ ]:
# ── Cell 6: TODO — Build LCEL RAG chain + guardrails ─────────────────────────
llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0, max_tokens=1024)

# --- Prompts ---
rag_prompt = ChatPromptTemplate.from_template("""\
You are the official HR Help Desk assistant for Zyro Dynamics Pvt. Ltd.
Answer employee HR questions STRICTLY using the provided policy documents.

CRITICAL RULES:
1. Use ONLY information from the Context — never invent or add external knowledge.
2. Be SPECIFIC: include exact numbers, durations, percentages, eligibility criteria, procedures.
3. If context clearly has the answer, give it completely and precisely.
4. If context does NOT contain sufficient information, say exactly:
   "This specific information is not covered in the available HR policy documents. Please contact HR directly."
5. Write in clear, professional English. Use bullet points for multi-part answers.

Context (from Zyro Dynamics HR Policy Documents):
{context}

Employee Question: {question}

HR Assistant Answer:""")

scope_prompt = ChatPromptTemplate.from_template("""\
You are a strict classifier. Is the following question related to company HR policies,
employment, leave, benefits, compensation, workplace conduct, IT security,
onboarding, travel expenses, or performance management at a company?

Answer ONLY with one word: YES or NO.

Question: {question}
Answer:""")

multi_query_prompt = ChatPromptTemplate.from_template("""\
Generate 3 different phrasings of the following HR question to improve document retrieval.
Output ONLY the 3 questions, one per line, no numbering, no extra text.

Original question: {question}

3 rephrased questions:""")

# --- HR keyword sets ---
HR_KEYWORDS = {
    'leave','vacation','pto','sick','maternity','paternity','wfh','remote','hybrid',
    'office','salary','ctc','bonus','increment','appraisal','performance','pip',
    'review','rating','probation','onboarding','offboarding','separation','resignation',
    'notice','travel','expense','reimbursement','policy','handbook','conduct',
    'harassment','posh','icc','device','security','data','confidential','employee',
    'hr','benefit','insurance','pf','gratuity','esic','holiday','overtime','shift',
    'allowance','deduction','tax','tds','grade','band','promotion','transfer','zyro',
    'dynamics','company','joining','full','final','settlement','termination',
    'disciplinary','code','ethics','misconduct','warning','annual','earned','casual',
    'compensatory','bereavement','accommodation','hotel','flight','cab','conveyance',
    'work','hours','attendance','payroll','stipend','internship',
}

OUT_OF_SCOPE_KEYWORDS = {
    'weather','cricket','football','sports','movie','recipe','cook','stock','crypto',
    'bitcoin','politics','government','news','celebrity','music','song','gaming',
    'restaurant','astrology','horoscope','joke','poem','story','doctor',
    'medical diagnosis','investment','share market','lottery','betting',
}

REFUSAL_MSG = (
    "I'm sorry, I can only answer HR-related questions based on Zyro Dynamics "
    "policy documents. This question appears to be outside my scope. "
    "Please contact HR directly for non-policy queries."
)

# --- Helpers ---
def is_hr_question(question):
    words = set(re.findall(r'\b\w+\b', question.lower()))
    if words & OUT_OF_SCOPE_KEYWORDS:
        return False
    if words & HR_KEYWORDS:
        return True
    return None

def format_docs(docs):
    seen, result = set(), []
    for doc in docs:
        c = doc.page_content.strip()
        if c not in seen:
            seen.add(c)
            result.append(c)
    return '\n\n---\n\n'.join(result)

def multi_query_retrieve(question, retriever):
    try:
        raw      = (multi_query_prompt | llm | StrOutputParser()).invoke({'question': question})
        variants = [q.strip() for q in raw.strip().split('\n') if q.strip()][:3]
    except Exception:
        variants = []
    seen_ids, docs = set(), []
    for q in [question] + variants:
        try:
            for doc in retriever.invoke(q):
                doc_id = hash(doc.page_content[:100])
                if doc_id not in seen_ids:
                    seen_ids.add(doc_id)
                    docs.append(doc)
        except Exception:
            pass
    return docs or retriever.invoke(question)

# --- Main RAG function ---
def ask_bot(question: str) -> dict:
    question = question.strip()

    # Layer 1: keyword fast-path
    keyword_result = is_hr_question(question)
    if keyword_result is False:
        return {'answer': REFUSAL_MSG, 'sources': []}

    # Layer 2: LLM scope classification for ambiguous questions
    if keyword_result is None:
        cls = (scope_prompt | llm | StrOutputParser()).invoke({'question': question}).strip().upper()
        if 'NO' in cls and 'YES' not in cls:
            return {'answer': REFUSAL_MSG, 'sources': []}

    # Layer 3: multi-query retrieval
    docs = multi_query_retrieve(question, base_retriever)

    if not docs:
        return {
            'answer': 'This specific information is not covered in the available HR policy documents. Please contact HR directly.',
            'sources': []
        }

    context = format_docs(docs)
    chain   = (
        {'context': RunnableLambda(lambda _: context), 'question': RunnablePassthrough()}
        | rag_prompt | llm | StrOutputParser()
    )
    answer  = chain.invoke(question)
    sources = sorted(set(doc.metadata.get('source', '') for doc in docs))
    return {'answer': answer, 'sources': sources}

print('✅ RAG chain with dual-layer guardrails ready')

In [ ]:
# ── Cell 7: TODO — Add out-of-scope guardrails (already in ask_bot above) ─────
# Test the guardrails
test_oos = ask_bot('What is the capital of France?')
print('Out-of-scope test:')
print(f'  Q: What is the capital of France?')
print(f'  A: {test_oos["answer"]}')
print()
assert 'only answer HR-related questions' in test_oos['answer'], 'Guardrail not working!'
print('✅ Guardrails working correctly')

In [ ]:
# ── Cell 8: Encoding helpers ─────────────────────────────────────────────────
def encode_text(text: str) -> str:
    return base64.b64encode(text.encode('utf-8')).decode('utf-8')

def decode_text(encoded: str) -> str:
    return base64.b64decode(encoded.encode('utf-8')).decode('utf-8')

print('✅ Encoding helpers ready')

In [ ]:
# ── Cell 9: PASTE ENCODED QUESTIONS HERE ─────────────────────────────────────
# Instructions:
# 1. Open the ORIGINAL starter notebook on Kaggle
# 2. Find the cell that defines the encoded questions (question_enc list)
# 3. Copy each encoded string and paste it below
# 4. The format is base64-encoded text

QUESTIONS_ENC = [
    # Paste your 15 encoded questions here as strings, in order Q01-Q15
    # Example:
    # 'V2hhdCBpcyB0aGUgYW5udWFsIGxlYXZlIHBvbGljeT8=',  # Q01
    # 'V2hhdCBpcyB0aGUgV0ZIIHBvbGljeT8=',              # Q02
    # ... and so on for Q03-Q15
]

# If you have them as a dict:
# QUESTIONS_ENC = {'Q01': '...', 'Q02': '...', ...}

if QUESTIONS_ENC:
    print(f'✅ {len(QUESTIONS_ENC)} encoded questions loaded')
    for i, enc in enumerate(QUESTIONS_ENC[:3], 1):
        print(f'  Q{i:02d}: {decode_text(enc)}')
else:
    print('⚠️  No questions loaded yet — paste encoded questions above')

In [ ]:
# ── Cell 10: Your Submission Links ───────────────────────────────────────────
STREAMLIT_URL = 'PASTE_YOUR_STREAMLIT_URL_HERE'
LANGSMITH_URL = 'PASTE_YOUR_LANGSMITH_TRACE_URL_HERE'

print(f'Streamlit : {STREAMLIT_URL}')
print(f'LangSmith : {LANGSMITH_URL}')

In [ ]:
# ── Cell 11: Run all 15 questions & collect answers ───────────────────────────
results = []

q_ids = [f'Q{i:02d}' for i in range(1, 16)]

for idx, q_enc in enumerate(QUESTIONS_ENC):
    qid      = q_ids[idx] if idx < len(q_ids) else f'Q{idx+1:02d}'
    question = decode_text(q_enc)

    print(f'\n{\'=\'*60}')
    print(f'🔍 {qid}: {question}')

    response   = ask_bot(question)
    answer     = response['answer']
    answer_enc = encode_text(answer)

    print(f'💬 {answer[:250]}...' if len(answer) > 250 else f'💬 {answer}')
    if response['sources']:
        src_names = [os.path.basename(s) for s in response['sources']]
        print(f'📄 Sources: {src_names}')

    results.append({
        'question_id':    qid,
        'question_enc':   q_enc,
        'answer_enc':     answer_enc,
        'streamlit_link': STREAMLIT_URL,
        'langsmith_link': LANGSMITH_URL,
    })

print(f'\n\n✅ Answered {len(results)} / 15 questions')

In [ ]:
# ── Cell 12: Sample test questions (as per competition Cell 12) ───────────────
print('\n📋 SAMPLE TEST QUESTIONS\n')
sample_questions = [
    'How many casual leaves are employees entitled to per year?',
    'What is the work from home policy for employees at Zyro Dynamics?',
    'What are the guidelines for business travel reimbursements?',
]
for q in sample_questions:
    print(f'Q: {q}')
    res = ask_bot(q)
    print(f'A: {res["answer"]}')
    print()

In [ ]:
# ── Cell 13: LangSmith trace URL instructions ─────────────────────────────────
print("""
📡 HOW TO GET YOUR LANGSMITH TRACE URL
======================================
1. Go to https://smith.langchain.com
2. Sign in → click 'Projects' in left sidebar
3. Open project: zyro-rag-challenge
4. Click on any trace → click 'Share' → copy public URL
5. Paste it in Cell 10 as LANGSMITH_URL
6. Re-run Cells 11 and 15 to regenerate submission.csv

URL format: https://smith.langchain.com/public/xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx/r
""")

In [ ]:
# ── Cell 14: Streamlit App reference ─────────────────────────────────────────
print(f"""
🚀 STREAMLIT APP
================
Live URL: {STREAMLIT_URL}
GitHub  : https://github.com/VanjariAbhinay/zyro-hr-bot

Features:
  • BAAI/bge-base-en-v1.5 embeddings
  • MMR retrieval (k=6, fetch_k=20)
  • Multi-query retrieval (3 variants)
  • Dual-layer out-of-scope guardrails
  • LangSmith tracing enabled
  • Source citation display
""")

In [ ]:
# ── Cell 15 (LAST): Generate submission.csv ───────────────────────────────────
if not results:
    print('❌ No results yet!')
    print('   1. Fill in QUESTIONS_ENC in Cell 9 with your encoded questions')
    print('   2. Fill in STREAMLIT_URL and LANGSMITH_URL in Cell 10')
    print('   3. Re-run Cell 11')
    print('   4. Then re-run this cell')
else:
    df = pd.DataFrame(results)
    df = df[['question_id', 'question_enc', 'answer_enc', 'streamlit_link', 'langsmith_link']]
    df.to_csv('submission.csv', index=False)

    print(f'✅ submission.csv generated — {len(df)} rows')
    print()
    print('Submission preview:')
    for _, row in df.iterrows():
        q_text = decode_text(row['question_enc'])
        a_text = decode_text(row['answer_enc'])
        scope  = '❌ OUT-OF-SCOPE (refused)' if 'only answer HR-related' in a_text else '✅ In-scope'
        print(f"  {row['question_id']}: [{scope}] {q_text[:55]}")

    print()
    print('⬇️  Download: File menu → Download → submission.csv')